# Analiza zahtjeva za troškove

Ovaj bilježnik demonstrira kako kreirati agente koji koriste dodatke za obradu troškova putovanja iz lokalnih slika računa, generiranje e-pošte za zahtjev troškova i vizualizaciju podataka o troškovima pomoću tortnog grafikona. Agenti dinamički biraju funkcije na temelju konteksta zadatka.

Koraci:  
1. OCR agent obrađuje lokalnu sliku računa i izvlači podatke o troškovima putovanja.  
2. Email agent generira e-poštu za zahtjev troškova.

### Primjer scenarija troškova putovanja:  
Zamislite da ste zaposlenik koji putuje na poslovni sastanak u drugi grad. Vaša tvrtka ima politiku nadoknade svih opravdanih troškova vezanih uz putovanje. Evo rasporeda potencijalnih troškova putovanja:  
- Prijevoz:  
Povratna avionska karta iz vašeg matičnog grada do odredišnog grada.  
Taksi ili usluge prijevoza do i od zračne luke.  
Lokalni prijevoz u odredišnom gradu (poput javnog prijevoza, unajmljenih automobila ili taksija).

- Smještaj:  
Boravak u hotelu tri noći u poslovnom hotelu srednje klase blizu mjesta sastanka.

- Obroci:  
Dnevni paušal za obroke za doručak, ručak i večeru, temeljen na politici tvrtke o dnevnicama.

- Razni troškovi:  
Naknade za parkiranje na aerodromu.  
Naknade za pristup internetu u hotelu.  
Bakšiš ili male usluge.

- Dokumentacija:  
Podnosite sve račune (avionske karte, taksi, hotel, obroci itd.) i ispunjeni izvještaj o troškovima za nadoknadu.


## Uvoz potrebnih biblioteka

Uvezite potrebne biblioteke i module za bilježnicu.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definiranje modela troškova

 Kreirajte Pydantic model za pojedinačne troškove i klasu ExpenseFormatter za pretvaranje korisničkog upita u strukturirane podatke o troškovima.

 Svaki trošak bit će predstavljen u formatu:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definiranje alata - Generiranje e-pošte

Kreirajte funkciju alata za generiranje e-pošte za podnošenje zahtjeva za naknadu troškova.  
- Ovaj alat koristi `@tool` dekorator iz Microsoft Agent Frameworka.  
- Izračunava ukupni iznos troškova i formatira detalje u tijelo e-pošte.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Alat za izdvajanje putnih troškova iz slika računa

Izradite funkcijsku alatku za izdvajanje putnih troškova iz slika računa.
- Ovaj alat koristi dekorator `@tool` iz Microsoft Agent Frameworka.
- Čita sliku računa, kodira je u base64 i vraća data URI agentu za analizu.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Obrada troškova

Definirajte agente i povežite ih u sekvencijalni tijek rada koristeći `WorkflowBuilder`.
- OCR agent izvlači strukturirane podatke o troškovima iz slike računa koristeći alat `load_receipt_image`.
- Email agent preuzima izdvojene podatke i generira profesionalni email za zahtjev troškova koristeći alat `generate_expense_email`.
- `WorkflowBuilder` s `add_edge` stvara sekvencijalni tijek: OCR agent → Email agent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Glavna funkcija

Izgradi sekvencijalni tijek rada i pokreni ga za obradu slike računa i generiranje e-pošte zahtjeva za nadoknadom troškova.


> **Napomena:** Ovaj tijek rada trenutno prosljeđuje sliku računa kao tekst u base64 formatu, što većina chat modela (uključujući gpt-4o) neće tretirati kao sliku.  
> Također može premašiti veličinu kontekstnog prozora modela. Preporučuje se pokretanje OCR-a s Azure AI Vision (ili nekim drugim OCR alatom) i prosljeđivanje samo izdvojenog teksta, ili refaktoriranje za slanje slike kao poruke `image_url`.  
> Ako želite samo izbjeći pogreške u kontekstu, pokušajte s manjom slikom računa ili modelom s većim kontekstnim prozorom.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
